# SilverGuard ML: next steps (from 6 Nov 2026) (Generated by AI)

Where the ML side is now, and what to do next. Tick boxes as you go.

**Where I am**
- Activity: UCI model, Capture-24 RF and a single `mag_std` threshold all score ~0.77 on held-out Capture-24 people. Features are the limit, not the model.
- Decision so far: `mag_std` threshold on the device, debounced walking/still for the dashboard, raw moving/still feeds the inactivity alert.
- Already have `har_model.h` and the ESP32 test sketch in `firmware/tests/`.

**Overall design:** models detect → rules decide → LLM talks. The alert/escalation stays a hard rule, never the LLM.

## 1. Close out the activity model

- [ ] Record own ESP32 sessions (walking, sitting, standing, typing, lying) with `tests/esp32_recordings/record_session.py`
- [ ] Set the `mag_std` threshold from those recordings (not from Capture-24 alone)
- [ ] Tune debounce `MIN_WALK_SEC` / `MIN_STILL_SEC` so labels don't flicker
- [ ] One experiment: richer accel-only features on Capture-24 (frequency content, wrist angle, longer windows) vs the threshold. Only switch if it clearly wins
- [ ] Test on GOTOV wrist data (35 adults over 60) and report the drop vs Capture-24

## 2. On-device event logic

Watch only sends events, never raw data (except debug mode).

```
every window:
  label = model(features)
  if fall candidate: run fall pipeline (section 3), send immediately if confirmed
  elif label stable 3 windows (~30 s) and != current: send "activity_change"
every 1-2 min: send heartbeat (alive + battery %)
every 30 min: send status (label + since when)
```

- [ ] Agree the event JSON with the team (Objective 3 depends on it), e.g. `{"event": "activity_change", "label": "resting", "since": "13:05", "battery": 62}`
- [ ] Rules (inactivity > 60-120 min daytime → check-in) live on the server, not the watch, so thresholds can change without reflashing
- [ ] Offline fallback on the watch: pre-recorded prompt + button, escalate on silence
- [ ] Keep a raw-data debug mode for recording datasets and evaluation

## 3. Fall detection (separate pipeline)

Agreed design:
1. **Candidate:** acc magnitude spike > A **and** gyro magnitude spike > G within ~0.5 s
2. Wait ~10 s and record
3. Post-spike movement (std) < S → voice "Are you okay?"; otherwise back to idle

- [ ] Download WEDA-FALL (training) and filter to wrist
- [ ] Tune A and G for recall first (≥95% of falls pass), then S. Keep S loose ("not walking normally", not "completely still")
- [ ] Compare the rule against a small RF on the same ingredients (acc peak, gyro peak, post-spike std, wrist angle change). Use the RF only if it clearly beats the rule
- [ ] Cross-dataset test on FallAllD and UMAFall (wrist sensor only)
- [ ] Split by participant, never by window
- [ ] Record own falls onto a mattress + near misses (flopping onto a sofa, arm hitting a table)

In [ ]:
# Starter: find fall candidates in one recording (acc in g, gyro in deg/s, fs in Hz)
import numpy as np

def fall_candidates(acc, gyro, fs=50, A=2.5, G=200, pair_window_s=0.5):
    acc_mag = np.linalg.norm(acc, axis=1)
    gyro_mag = np.linalg.norm(gyro, axis=1)
    acc_idx = np.where(acc_mag > A)[0]
    gyro_idx = np.where(gyro_mag > G)[0]
    w = int(pair_window_s * fs)
    hits = [i for i in acc_idx if np.any(np.abs(gyro_idx - i) <= w)]
    # merge hits closer than 2 s into one candidate
    merged = []
    for i in hits:
        if not merged or i - merged[-1] > 2 * fs:
            merged.append(i)
    return merged

def post_spike_std(acc, idx, fs=50, delay_s=2, length_s=8):
    acc_mag = np.linalg.norm(acc, axis=1)
    seg = acc_mag[idx + int(delay_s * fs): idx + int((delay_s + length_s) * fs)]
    return seg.std() if len(seg) else np.nan

## 4. Put models on the ESP32

- [ ] Keep forests small (10-20 trees, `max_depth` ~8) and compare accuracy before/after shrinking
- [ ] Export with `micromlgen` or `emlearn` → `.h` header
- [ ] **Feature parity test:** run the same recording through Python `features.py` and the ESP32 C code; features and labels must match exactly
- [ ] Check timing and RAM on the C3 (single core, no FPU, also running WiFi + audio)

## 5. Hardware settings ML needs

- Accelerometer ±8 g minimum (±2 g clips impacts), ideally ±16 g
- Gyro ±2000 deg/s (default ±250 clips falls); keep gyro on if used for falls
- Sample at 50 Hz or faster, same rate as training data (resample otherwise)
- Units in g, sensor fixed firmly in the enclosure

## 6. Evaluation (Objective 4)

- Activity: balanced accuracy per participant on Capture-24, GOTOV and own recordings
- Falls: recall and false positives on daily activities; false alarms per day on Capture-24 (accel-only version)
- End-to-end: staged falls in the demo setup, time from fall to voice prompt
- Limitation to report: all fall datasets use simulated falls by younger adults

## Dataset links

| Dataset | Use | Link |
|---|---|---|
| Capture-24 | Activity training | https://github.com/OxWearables/capture24 |
| GOTOV | Activity test on over-60s | https://data.4tu.nl/datasets/f9bae0cd-ec4e-4cfb-aaa5-41bd1c5554ce |
| WEDA-FALL | Fall training (wrist, elderly ADLs) | https://github.com/joaojtmarques/WEDA-FALL |
| FallAllD | Fall cross-dataset test | https://ieee-dataport.org/open-access/fallalld-comprehensive-dataset-human-falls-and-activities-daily-living |
| UMAFall | Fall cross-dataset test | https://figshare.com/articles/dataset/UMA_ADL_FALL_Dataset_zip/4214283 |

FallAllD and UMAFall have several body locations: filter for the wrist sensor.